In [2]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [3]:
train_data = pd.read_csv("./data/samsum-train.csv")
validation_data = pd.read_csv("./data/samsum-validation.csv")

In [4]:
train_data.head

<bound method NDFrame.head of              id                                           dialogue  \
0      13818513  Amanda: I baked  cookies. Do you want some?\r\...   
1      13728867  Olivia: Who are you voting for in this electio...   
2      13681000  Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...   
3      13730747  Edward: Rachel, I think I'm in ove with Bella....   
4      13728094  Sam: hey  overheard rick say something\r\nSam:...   
...         ...                                                ...   
14727  13863028  Romeo: You are on my ‘People you may know’ lis...   
14728  13828570  Theresa: <file_photo>\r\nTheresa: <file_photo>...   
14729  13819050  John: Every day some bad news. Japan will hunt...   
14730  13828395  Jennifer: Dear Celia! How are you doing?\r\nJe...   
14731  13729017  Georgia: are you ready for hotel hunting? We n...   

                                                 summary  
0      Amanda baked cookies and will bring Jerry some...  
1      Oliv

In [7]:
train_data.shape

(14732, 3)

In [8]:
validation_data.shape

(818, 3)

In [9]:
# random sampling and reduce data for now(not compulsary)
train_data = train_data.sample(n=5000, random_state = 42).reset_index(drop=True)
compulsoryvalidation_data = validation_data.sample(n=500, random_state = 42).reset_index(drop=True)

In [10]:
# Pre-processing Data 
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",  text)  # remove next line etc.
    text = re.sub(r"s+", " ", text)  # extra space
    text = re.sub(r"<.*?>", " ", text)  # HTML Tags
    text = text.strip().lower()
    return text

In [11]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

In [12]:
train_data["dialogue"][0]

"violet: hi! i came acro  thi  au tin'  article and i thought that you might find it intere ting violet:   claire: hi! :) thank , but i've already read it. :) claire: but thank  for thinking about me :)"

In [13]:
# Tokenizar
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [15]:
# row data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_lenght = 512, truncation = True)
    targets = tokenizer(data["summary"], padding="max_length", max_lenght = 150, truncation = True)

    inputs["labels"] = targets["input_ids"]  # token id => add to input as labels
    return inputs

In [31]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
validation_dataset = validation_data.apply(tokenize, axis=1).tolist()


In [32]:
train_dataset[0]

{'input_ids': [2774, 9, 10, 1568, 140, 3, 7159, 8355, 3, 2444, 4752, 172, 3, 354, 83, 23, 9, 10, 103, 25, 214, 3, 1927, 15, 102, 36, 3745, 58, 36, 11488, 10, 3, 23, 214, 376, 55, 2774, 9, 10, 150, 855, 2774, 9, 10, 3, 23, 3, 5398, 3, 23, 3534, 83, 26, 58, 3, 354, 83, 23, 9, 10, 1728, 428, 376, 3, 9, 653, 55, 14981, 28, 703, 3, 32, 1621, 5, 36, 11488, 10, 703, 3, 32, 1621, 3, 2, 519, 3, 2, 519, 2774, 9, 10, 8957, 6, 2763, 3, 55, 3, 354, 83, 23, 9, 10, 3, 23, 31, 51, 3, 1462, 25, 31, 195, 333, 34, 55, 36, 11488, 10, 17945, 34, 31, 3, 9, 1123, 3, 7159, 2774, 9, 10, 3, 2, 519, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [33]:
#  inputs_ids
# 1 => EOS    0 => Padding
#  attention_mask => those have 1 its mean this important and valid values other not imp during the training
#  labels - target => summary token

In [34]:
 #  Working with our Model
#NLP => generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [35]:
import torch 
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device: ", device)
model.to(device)

Device:  mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [36]:
# Training Arguments for Transformer

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500  # 0 => learning rate default
)

In [38]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset
)

In [40]:
trainer.train()

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: MPS backend out of memory (MPS allocated: 8.99 GiB, other allocations: 72.75 MiB, max allowed: 9.07 GiB). Tried to allocate 64.00 MiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).